## make-convert_Tamura_raw:
### Convert Tamura data from raw file to netcdf

In [1]:

import numpy as np 
import os
import sys
import xarray as xr
import scipy.io as sio
import matplotlib.pyplot as plt
import datetime
from scipy.interpolate import griddata
from scipy.signal import convolve2d
import pandas as pd







# Time settings for IAF
year_start =  1992
year_end = 2017 # run from year_start up to the beginning of year year_end

# set paths
run ='TSDM_2hb'
os.chdir('/g/data/jk72/deg581/amery-hires-setup/notebooks')
proj_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
data_dir = os.path.join(proj_dir,'data')
src_dir = os.path.join(proj_dir,'src')
sys.path.append(src_dir)

# import socket
# comp_name = socket.gethostname()
# if comp_name=='SEES-3PV4VV3':
#     T_mask_path = os.path.join(data_dir,'raw','tamura','EASE_landmask_H.data')
#     T_lat_lon_path = os.path.join(data_dir,'raw','tamura','EASE_latlon_H.data')
#     T_data_path = os.path.join(data_dir,'raw','tamura')
#     era_path = os.path.join(data_dir,'raw','era_interim','ERA_Interim_1992_2011.2daily.*winds.nc')
#     R_grid_path = os.path.join(data_dir,'proc',run+'_v11_grd.nc')
# else:
T_mask_path = os.path.join('/g/data/jk72/iomp/obs/Tamura_daily/EASE_landmask_H.data')
T_lat_lon_path = os.path.join('/g/data/jk72/iomp/obs/Tamura_daily/EASE_latlon_H.data')
T_data_path = os.path.join('/g/data/jk72/iomp/obs/Tamura_daily/')


from ext.tools.NDinterp import NDinterp
from ext.tools.log_progress import log_progress

In [29]:
#read in tamura land mask
with open(T_mask_path,'rb') as fid:
    T_mask = np.fromfile(fid,count=(721*721),dtype='float32').reshape((721,721))
    T_mask = np.flipud(T_mask)

In [30]:
#get tamura lat lon coordinates
with open(T_lat_lon_path,'rb') as fid:
    T_lat_lon = np.fromfile(fid,count=(721*721*2),dtype='float32').reshape((2,721,721))
T_lat,T_lon = (T_lat_lon[0],T_lat_lon[1])
T_lat = np.flipud(T_lat)
T_lon = np.flipud(T_lon)
T_lon[T_lon<0]+=360

Now begin working through and saving netcdf files

In [49]:
month_nb

'02'

In [50]:
for working_year in range(year_start,year_end):
    print(working_year)
    out_file = os.path.join(data_dir,'proc',run+'_'+str(working_year)+'_data.nc')
    
    
    # MAKE THE MAGIC HAPPEN FOR EACH YEAR, AND SAVE TO INTERIM FILES
    month = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
    month_nb = ['01','02','03','04','05','06','07','08','09','10','11','12']
    daysPerMonth = [31,28,31,30,31,30,31,31,30,31,30,31]


    for month,days,month_nb in zip(month,daysPerMonth,month_nb):
        
        print('Processing month: ',month,'with days: ',days)

        # Get start and end of the month
        start_date = pd.Timestamp(f"{working_year}-{month_nb}-01")
        # use MonthEnd(1) to get last day of month
        end_date = (start_date + pd.offsets.MonthEnd(1))
    
        # Generate time vector for this month
        dates = pd.date_range(start=start_date, end=end_date, freq='D')
    
        # Optional: remove Feb 29 if you want only 365-day years
        if len(dates) == 29 and month_nb == '02':
            dates = dates[:-1]
        
        daysOfYear = dates #np.arange(dayOfYear,dayOfYear+days,dtype=int)
        
        print('Containing days of year: ',daysOfYear)

        # preparing empty dataset
        ds = xr.Dataset({'shflux_raw':(['shf_time','y','x'], np.empty((days,T_lon.shape[0],T_lon.shape[1]))),
                        'swflux_raw':(['swf_time','y','x'], np.empty((days,T_lon.shape[0],T_lon.shape[1]))),
                        'shflux_mod':(['shf_time','y','x'], np.empty((days,T_lon.shape[0],T_lon.shape[1]))),
                        'swflux_mod':(['swf_time','y','x'], np.empty((days,T_lon.shape[0],T_lon.shape[1])))},
                    coords={'shf_time':(['shf_time'],daysOfYear),
                            'swf_time':(['swf_time'],daysOfYear)})
        ds.shf_time.encoding["units"] = "seconds since 1970-01-01 00:00:00"
        ds.shf_time.encoding["calendar"] = "standard"
        ds.swf_time.encoding["units"] = "seconds since 1970-01-01 00:00:00"
        ds.swf_time.encoding["calendar"] = "standard"
    
        #open Tamura month flux data 
        T_month_path = os.path.join(T_data_path,'TSDM2hb_2007_'+month+'.data')
        with open(T_month_path,'rb') as fid:
            T_data = np.swapaxes(np.fromfile(fid,count = days*6*721*721 ,dtype='float32').reshape(days,6,721,721),0,1)
        
        #looping over the days with running day-of-the-year and day-of-the-month index
        for Tidx in np.arange(days):
            
            #read in Tamura heat and fresh water flux and turn in right position
            shflux_tmp = np.flipud(T_data[0,Tidx])
            ssflux_tmp = np.flipud(T_data[2,Tidx])


            
            #fill in tamuar mask for later resampling
            shflux_tmp[T_mask==0] = np.nan
            # shflux_tmp = NDinterp(shflux_tmp)
            ssflux_tmp[T_mask==0] = np.nan
            # ssflux_tmp = NDinterp(ssflux_tmp)
            
            ds.shflux_raw[Tidx] = shflux_tmp # save raw data
            ds.swflux_raw[Tidx] = ssflux_tmp # save raw data
            
            #now modified the variables.
            shflux_tmp[shflux_tmp > 0.0]*=0.5
            
            ds.shflux_mod[Tidx] = shflux_tmp # save modified data
            del shflux_tmp
            
            #convert to freshwater flux with convention positive up 'swf (E-P)',
            #that means a positive freshwater flux value results in positive salt flux value
            #and save to dataset
            refSalt = 34.4
            # original units of ssflux has units of Salt*[E-P] [m/day]
            # So to convert to correct units [m/s], need to multiply by 1/86400, as 1/86400 days per second,
            # # to go from 1/day to 1/s
            
            ds.swflux_mod[Tidx] = ssflux_tmp/refSalt*100 * 0.01/86400  # save modified data
            del ssflux_tmp
            
        #add attributes to data set and data arrays
        ds.attrs={'title':'Takeshi Tamura-derived raw SIP, and surface heat/fresh water fluxes ',
                            'date':str(datetime.date.today()),
                            'tamura_file':T_data_path}
        ds.shflux_mod.attrs = {'long_name': 'ROMS-modified surface net heat flux', 'units': 'Watts meter-2'}
        ds.swflux_mod.attrs = {'long_name': 'ROMS-modified surface freshwater flux (E-P)*Salt_ref',
                        'negative': 'net precipitation',
                        'positive': 'net evaporation',
                        'units': 'metre second-1',
                        '_FillValue':1e+37}
        ds.shflux_raw.attrs = {'long_name': 'surface net heat flux', 'units': 'Watts meter-2'}
        ds.swflux_raw.attrs = {'long_name': 'surface freshwater flux (E-P)',
                        'negative': 'net precipitation',
                        'positive': 'net evaporation',
                        'units': 'metre day-1',
                        '_FillValue':1e+37}        
        ds.shf_time.attrs = {'long_name': 'surface heat flux time'}
        ds.swf_time.attrs = {'long_name': 'surface freshwater flux time'}
        
        #save month as netcdf files
        for var,dim in zip(['shflux_mod','swflux_mod','shflux_raw','swflux_raw'],['shf_time','swf_time','shf_time','swf_time']):
            int_path = os.path.join(data_dir,'cache',run+'_'+var+'_'+month_nb+'.nc')
            print("Saving month to "+int_path)
            ds[var].to_netcdf(int_path,'w',unlimited_dims=dim)
        #del ds
        
        #update the day of the year value for next month
        dayOfYear += days
        
        
    #collect all interim results, merge to yearly data amd adjust cycle length attribute
    #save forcing files in processed folder
    datasets = []
    for var,dim in zip(['shflux_mod','swflux_mod','shflux_raw','swflux_raw'],['shf_time','swf_time','shf_time','swf_time']):
        ds = xr.open_mfdataset(os.path.join(data_dir,'cache',run+'_'+var+'_??.nc'))
        datasets.append(ds)
        # ds[dim]
        # ds[dim].attrs['cycle_length'] = float(365)
    combined_ds = xr.merge(datasets)    
    out_path = os.path.join(data_dir,'proc',run+'_'+str(working_year)+'.nc')
    print('saving final to'+out_path)
    combined_ds.to_netcdf(out_path,'w')
    del combined_ds,datasets

1992
Processing month:  jan with days:  31
Containing days of year:  DatetimeIndex(['1992-01-01', '1992-01-02', '1992-01-03', '1992-01-04',
               '1992-01-05', '1992-01-06', '1992-01-07', '1992-01-08',
               '1992-01-09', '1992-01-10', '1992-01-11', '1992-01-12',
               '1992-01-13', '1992-01-14', '1992-01-15', '1992-01-16',
               '1992-01-17', '1992-01-18', '1992-01-19', '1992-01-20',
               '1992-01-21', '1992-01-22', '1992-01-23', '1992-01-24',
               '1992-01-25', '1992-01-26', '1992-01-27', '1992-01-28',
               '1992-01-29', '1992-01-30', '1992-01-31'],
              dtype='datetime64[ns]', freq='D')
Saving month to /g/data/jk72/deg581/amery-hires-setup/data/cache/TSDM_2hb_shflux_mod_01.nc
Saving month to /g/data/jk72/deg581/amery-hires-setup/data/cache/TSDM_2hb_swflux_mod_01.nc
Saving month to /g/data/jk72/deg581/amery-hires-setup/data/cache/TSDM_2hb_shflux_raw_01.nc
Saving month to /g/data/jk72/deg581/amery-hires-setup/d

In [25]:
ds_sw = xr.open_mfdataset(
    files,
    drop_variables=['shflux_mod', 'shflux_raw', 'shf_time', 'swflux_raw'],
    combine='by_coords',
    coords='minimal',
    compat='override'
)

ds_sh = xr.open_mfdataset(
    files,
    drop_variables=['swflux_mod', 'swf_time','shflux_raw','swflux_raw'],
    combine='by_coords',
    coords='minimal',
    compat='override'
)

# Merge manually
ds = xr.merge([ds_sw, ds_sh], compat='override')

In [29]:
print(ds)

<xarray.Dataset> Size: 76GB
Dimensions:     (swf_time: 9125, y: 721, x: 721, shf_time: 9125)
Coordinates:
  * swf_time    (swf_time) datetime64[ns] 73kB 1992-01-01 ... 2016-12-31
  * shf_time    (shf_time) datetime64[ns] 73kB 1992-01-01 ... 2016-12-31
Dimensions without coordinates: y, x
Data variables:
    swflux_mod  (swf_time, y, x) float64 38GB dask.array<chunksize=(365, 721, 721), meta=np.ndarray>
    shflux_mod  (shf_time, y, x) float64 38GB dask.array<chunksize=(365, 721, 721), meta=np.ndarray>


In [27]:
ds.nbytes/1e9

75.896932